<a href="https://colab.research.google.com/github/alexander-myronov/tf_custom_gpu_op/blob/master/AlphaFold2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold: AlphaFold2 w/ MMseqs2

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold - Making protein folding accessible to all.
*bioRxiv*, 2021](https://www.biorxiv.org/content/10.1101/2021.08.15.456425v2)

- This notebook provides basic functionality for **protein structure (Alphafold2)** and **complex prediction (Alphafold2-multimer)**. Advanced features such as recycles, sampling, ... ->[advanced notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/beta/AlphaFold2_advanced.ipynb).
- This notebook replaces the homology detection and MSA pairing of AlphaFold2 with MMseqs2. For a comparision against the [Deepmind Colab](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb) and the full [AlphaFold2](https://github.com/deepmind/alphafold) system read our [preprint](https://www.biorxiv.org/content/10.1101/2021.08.15.456425v1).

Old versions: [v1.0-alpha](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.0-alpha/AlphaFold2.ipynb), [v1.1-permultimer](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.1-premultimer/AlphaFold2.ipynb)

For more details, see **<a href="#Instructions">bottom</a>** of the notebook and checkout the **[ColabFold GitHub](https://github.com/sokrypton/ColabFold)**.

In [3]:
!rm test_3e7e8.custom.a3m

In [42]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`
from google.colab import files
import os.path
import re
import hashlib

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

query_sequence = 'QKEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPELIMFIYQAGDKEDGRFTAQLNKASQYVSLLIRDSQPSDSATYLCAAVIGNAQGGMLTFGTGTRLTIIP:KAGVTQTPRYLIKTRGQQVTLSCSPIPGHRAVSWYQQTPGQGLQFLFEYVHGEERNKGNFPGRFSGRQFSNSRSEMNVSTLELGDSALYLCASSPWDSPNVQYFGPGTRLTVT:SLLQHLIGL:GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQ' #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a mono-dimer

# remove whitespaces
query_sequence = "".join(query_sequence.split())

jobname = 'test2' #@param {type:"string"}
# remove whitespaces
jobname = "".join(jobname.split())
jobname = re.sub(r'\W+', '', jobname)
jobname = add_hash(jobname, query_sequence)


with open(f"{jobname}.csv", "w") as text_file:
    text_file.write(f"id,sequence\n{jobname},{query_sequence}")

queries_path=f"{jobname}.csv"

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef only)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 2 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
use_msa = True if msa_mode.startswith("MMseqs2") else False
use_env = True if msa_mode == "MMseqs2 (UniRef+Environmental)" else False
use_amber = False #@param {type:"boolean"}
use_templates = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### Experimental options
save_to_google_drive = False #@param {type:"boolean"}
#@markdown ---
#@markdown Don't forget to hit `Runtime` -> `Run all` after updating the form.

with open(f"{jobname}.log", "w") as text_file:
    text_file.write("num_models=%s\n" % num_models)
    text_file.write("use_amber=%s\n" % use_amber)
    text_file.write("use_msa=%s\n" % use_msa)
    text_file.write("msa_mode=%s\n" % msa_mode)
    text_file.write("use_templates=%s\n" % use_templates)

# decide which a3m to use
if use_msa:
  a3m_file = f"{jobname}.a3m"
elif msa_mode == "custom":
  a3m_file = f"{jobname}.custom.a3m"
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    header = 0
    import fileinput
    for line in fileinput.FileInput(custom_msa,inplace=1):
      if line.startswith(">"):
         header = header + 1
      if line.startswith("#"):
        continue
      if not line.rstrip():
        continue
      if line.startswith(">") == False and header == 1:
         query_sequence = line.rstrip()
      print(line, end='')

    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
    print(f"moving {custom_msa} to {a3m_file}")
else:
  a3m_file = f"{jobname}.single_sequence.a3m"
  with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

if save_to_google_drive:
  from pydrive.drive import GoogleDrive
  from pydrive.auth import GoogleAuth
  from google.colab import auth
  from oauth2client.client import GoogleCredentials
  auth.authenticate_user()
  gauth = GoogleAuth()
  gauth.credentials = GoogleCredentials.get_application_default()
  drive = GoogleDrive(gauth)
  print("You are logged into Google Drive and are good to go!")

In [17]:
!pip install -q pydrive

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 34.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [18]:
from pydrive.drive import GoogleDrive
from pydrive.auth import GoogleAuth
from google.colab import auth
from oauth2client.client import GoogleCredentials
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
print("You are logged into Google Drive and are good to go!")

You are logged into Google Drive and are good to go!


In [43]:
#@title Install dependencies
%%bash -s $use_amber $use_msa $use_templates

set -e

USE_AMBER=$1
USE_MSA=$2
USE_TEMPLATES=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  pip install -q biopython dm-haiku ml-collections py3Dmol
  # Trick for dev stage because otherwise pip won't install newer git versions
  pip uninstall -y -q colabfold
  pip install -q "colabfold[alphafold]==1.6.3" # Changed for release notebook
  touch COLABFOLD_READY
fi

# download libraries for interfacing with MMseqs2 API
if [ ${USE_MSA} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f MMSEQ2_READY ]; then
    apt-get -qq -y update 2>&1 1>/dev/null
    apt-get -qq -y install jq curl zlib1g gawk 2>&1 1>/dev/null
    touch MMSEQ2_READY
  fi
fi
# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    bash Miniconda3-latest-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniconda3-latest-Linux-x86_64.sh
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign3=3.2.2 hhsuite=3.3.0 python=3.7 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=7.5.1 python=3.7 pdbfixer 2>&1 1>/dev/null
  wget -qnc https://raw.githubusercontent.com/deepmind/alphafold/main/docker/openmm.patch
  (cd /usr/local/lib/python3.7/site-packages; patch -s -p0 < /content/openmm.patch)
  wget -qnc https://git.scicore.unibas.ch/schwede/openstructure/-/raw/7102c63615b64735c4941278d92b554ec94415f8/modules/mol/alg/src/stereo_chemical_props.txt
  touch AMBER_READY
fi


CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    - https://repo.anaconda.com/pkgs/main
    - https://repo.anaconda.com/pkgs/r

To accept these channels' Terms of Service, run the following commands:
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

For information on safely removing channels from your conda configuration,
please see the official documentation:

    https://www.anaconda.com/docs/tools/working-with-conda/channels



CalledProcessError: Command 'b'\nset -e\n\nUSE_AMBER=$1\nUSE_MSA=$2\nUSE_TEMPLATES=$3\n\nif [ ! -f COLABFOLD_READY ]; then\n  # install dependencies\n  pip install -q biopython dm-haiku ml-collections py3Dmol\n  # Trick for dev stage because otherwise pip won\'t install newer git versions\n  pip uninstall -y -q colabfold\n  pip install -q "colabfold[alphafold]==1.6.3" # Changed for release notebook\n  touch COLABFOLD_READY\nfi\n\n# download libraries for interfacing with MMseqs2 API\nif [ ${USE_MSA} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then\n  if [ ! -f MMSEQ2_READY ]; then\n    apt-get -qq -y update 2>&1 1>/dev/null\n    apt-get -qq -y install jq curl zlib1g gawk 2>&1 1>/dev/null\n    touch MMSEQ2_READY\n  fi\nfi\n# setup conda\nif [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then\n  if [ ! -f CONDA_READY ]; then\n    wget -qnc https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh\n    bash Miniconda3-latest-Linux-x86_64.sh -bfp /usr/local 2>

In [ ]:
#@title Run Prediction

import sys

from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run

from pathlib import Path

# For some reason we need that to get pdbfixer to import
if use_amber and '/usr/local/lib/python3.7/site-packages/' not in sys.path:
    sys.path.insert(0, '/usr/local/lib/python3.7/site-packages/')

result_dir="."
setup_logging(Path(".").joinpath("log.txt"))
queries, is_complex = get_queries(queries_path)
# download_alphafold_params("alphafold2_multimer_v3", Path("."))


In [33]:
!ls ./

cite.bibtex
COLABFOLD_READY
config.json
log.txt
MMSEQ2_READY
output
params
sample_data
test2_3e7e8.csv
test2_3e7e8.log
test_3e7e8.csv
test_3e7e8.custom.a3m
test_3e7e8.custom_coverage.png
test_3e7e8.custom.done.txt
test_3e7e8.custom_pae.png
test_3e7e8.custom_plddt.png
test_3e7e8.custom_predicted_aligned_error_v1.json
test_3e7e8.custom_scores_rank_001_alphafold2_multimer_v3_model_1_seed_000.json
test_3e7e8.custom_scores_rank_001_alphafold2_multimer_v3_model_3_seed_000.json
test_3e7e8.custom_scores_rank_001_alphafold2_multimer_v3_model_4_seed_000.json
test_3e7e8.custom_scores_rank_002_alphafold2_multimer_v3_model_1_seed_000.json
test_3e7e8.custom_scores_rank_002_alphafold2_multimer_v3_model_5_seed_000.json
test_3e7e8.custom_scores_rank_003_alphafold2_multimer_v3_model_1_seed_000.json
test_3e7e8.custom_scores_rank_003_alphafold2_multimer_v3_model_3_seed_000.json
test_3e7e8.custom_scores_rank_004_alphafold2_multimer_v3_model_2_seed_000.json
test_3e7e8.custom_scores_rank_004_alphafold2_multi

In [44]:
run(
    queries=queries,
    result_dir=result_dir,
    model_type="alphafold2_multimer_v3",
    use_templates=use_templates,
    use_amber=use_amber,
    msa_mode=msa_mode,
    num_models=1,#num_models,
    num_recycles=num_recycles,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    recompile_padding=1.0,
    rank_mode="auto",
    pair_mode="unpaired_paired",
    stop_at_score=float(100),
)

2026-10-08 11:47:26,903 Running on GPU
2026-10-08 11:47:26,908 Found 7 citations for tools or databases
2026-10-08 11:47:26,908 Query 1/1: test2_3e7e8 (length 415)
2026-10-08 11:47:26,908 No user agent specified. Please set a user agent (e.g., 'toolname/version contact@email') to help us debug in case of problems. This warning will become an error in the future.
2026-10-08 11:47:53,484 Sequence 0 found templates: ['6mkr_A', '6mkd_A', '7pdx_CCC', '4x6b_A', '4x6d_E', '4x6b_C', '4x6c_E', '4x6d_G', '6xqp_G', '6amu_D', '6xqp_E', '6am5_D', '3qdj_D', '6vqo_D', '3qeu_A', '5sws_D', '6xqq_A', '6vqo_H', '6xqq_C', '7na5_D']
2026-10-08 11:48:04,795 Sequence 1 found templates: ['6uln_E', '5nmd_B', '4jry_E', '1kgc_E', '6eh4_E', '6fr4_B', '1u3h_F', '4p4k_H', '1zgl_P', '2z35_B', '1zgl_R', '1zgl_T', '2jcc_F', '4qrr_E', '6xc9_J', '5bs0_E', '4onh_B', '7phr_B', '6vrm_E', '6uk4_E']
2026-10-08 11:48:04,796 Sequence 2 found no templates
2026-10-08 11:48:18,474 Sequence 3 found templates: ['6rpb_K', '5jzi_F', 

/usr/local/lib/python3.13/dist-packages/alphafold/model/modules_multimer.py:114: UserWarning: Explicitly requested dtype int64 requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, logits.shape, axis)


2026-10-08 11:49:45,077 alphafold2_multimer_v3_model_1_seed_000 recycle=0 pLDDT=89.7 pTM=0.738 ipTM=0.657
2026-10-08 11:50:25,490 alphafold2_multimer_v3_model_1_seed_000 recycle=1 pLDDT=92.2 pTM=0.861 ipTM=0.828 tol=0.612
2026-10-08 11:51:06,274 alphafold2_multimer_v3_model_1_seed_000 recycle=2 pLDDT=92.5 pTM=0.865 ipTM=0.832 tol=0.308
2026-10-08 11:51:06,276 alphafold2_multimer_v3_model_1_seed_000 took 162.5s (2 recycles)
2026-10-08 11:51:06,338 reranking models by 'multimer' metric
2026-10-08 11:51:06,338 rank_001_alphafold2_multimer_v3_model_1_seed_000 pLDDT=92.5 pTM=0.865 ipTM=0.832 ipSAE=A-B:0.716,A-C:0.393,A-D:0.643,B-C:0.302,B-D:0.474,C-D:0.813 pDockQ2=A-B:0.454,A-C:0.143,A-D:0.352,B-C:0.0982,B-D:0.306,C-D:0.699
2026-10-08 11:51:06,819 Done


{'rank': [['rank_001_alphafold2_multimer_v3_model_1_seed_000']],
 'metric': [[{'mean_plddt': 92.5,
    'ptm': 0.86474609375,
    'iptm': 0.83203125,
    'print_line': ' pLDDT=92.5 pTM=0.865 ipTM=0.832 ipSAE=A-B:0.716,A-C:0.393,A-D:0.643,B-C:0.302,B-D:0.474,C-D:0.813 pDockQ2=A-B:0.454,A-C:0.143,A-D:0.352,B-C:0.0982,B-D:0.306,C-D:0.699'}]]}

In [45]:

!ls -lh

total 40M
-rw-r--r-- 1 root root  3.0K Oct  8 11:47 cite.bibtex
-rw-r--r-- 1 root root     0 Oct  8 10:37 COLABFOLD_READY
-rw-r--r-- 1 root root     0 Oct  8 11:47 CONDA_READY
-rw-r--r-- 1 root root  1.2K Oct  8 11:47 config.json
-rw-r--r-- 1 root root  4.9K Oct  8 11:51 log.txt
-rw-r--r-- 1 root root     0 Oct  8 11:32 MMSEQ2_READY
drwxr-xr-x 2 root root  4.0K Oct  8 10:54 output
drwxr-xr-x 2 root root  4.0K Oct  8 10:41 params
drwxr-xr-x 1 root root  4.0K Sep 30 13:28 sample_data
-rw-r--r-- 1 root root  8.3M Oct  8 11:48 test2_3e7e8.a3m
drwxr-xr-x 5 root root  4.0K Oct  8 11:47 test2_3e7e8_all
-rw-r--r-- 1 root root  109K Oct  8 11:48 test2_3e7e8_coverage.png
-rw-r--r-- 1 root root   442 Oct  8 11:46 test2_3e7e8.csv
-rw-r--r-- 1 root root     0 Oct  8 11:51 test2_3e7e8.done.txt
-rw-r--r-- 1 root root    92 Oct  8 11:46 test2_3e7e8.log
-rw-r--r-- 1 root root  100K Oct  8 11:51 test2_3e7e8_pae.png
drwxr-xr-x 2 root root  4.0K Oct  8 11:37 test2_3e7e8_pairgreedy
-rw-r--r-- 1 root root  

In [47]:
#@title Display 3D structure {run: "auto"}
import py3Dmol
import glob
import matplotlib.pyplot as plt
rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
color = "chain" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = True #@param {type:"boolean"}
show_mainchains = True #@param {type:"boolean"}

jobname_prefix = ".custom" if msa_mode == "custom" else ""
if use_amber:
  pdb_filename = f"{jobname}{jobname_prefix}_relaxed_model_*_rank_{rank_num}.pdb"
else:
  pdb_filename = f"{jobname}{jobname_prefix}_unrelaxed_rank_{rank_num:03d}_*.pdb"

pdb_file = glob.glob(pdb_filename)
# pdb_file = ["test_3e7e8.custom_unrelaxed_rank_005_alphafold2_multimer_v3_model_5_seed_000.pdb"]
print(pdb_file)

def show_pdb(rank_num=1, show_sidechains=False, show_mainchains=False, color="lDDT"):
  model_name = f"rank_{rank_num}"
  if use_amber:
    pdb_filename = f"{jobname}{jobname_prefix}_relaxed_{model_name}.pdb"
  else:
    pdb_filename = f"{jobname}{jobname_prefix}_unrelaxed_{model_name}.pdb"

  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file[0],'r').read(),'pdb')

  if color == "lDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    for n,chain,color in zip(range(4),list("ABCDEFGH"),
                     ["lime","cyan","magenta","yellow","salmon","white","blue","orange"]):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  view.zoomTo()
  return view


show_pdb(rank_num,show_sidechains, show_mainchains, color).show()

# from IPython.display import Image
# from IPython.display import display

# pae = Image(f"{jobname}{jobname_prefix}_PAE.png")
# plddt = Image(f"{jobname}{jobname_prefix}_coverage_lDDT.png")
# display(pae, plddt)



['test2_3e7e8_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb']


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
!ls -lh

total 28M
-rw-r--r-- 1 root root 1020 Oct  8 11:09 cite.bibtex
-rw-r--r-- 1 root root    0 Oct  8 10:37 COLABFOLD_READY
-rw-r--r-- 1 root root 1.2K Oct  8 11:09 config.json
-rw-r--r-- 1 root root 3.2K Oct  8 11:23 log.txt
drwxr-xr-x 2 root root 4.0K Oct  8 10:54 output
drwxr-xr-x 2 root root 4.0K Oct  8 10:41 params
drwxr-xr-x 1 root root 4.0K Sep 30 13:28 sample_data
-rw-r--r-- 1 root root  441 Oct  8 11:07 test_3e7e8.csv
-rw-r--r-- 1 root root  13M Oct  8 11:09 test_3e7e8.custom.a3m
-rw-r--r-- 1 root root  85K Oct  8 11:09 test_3e7e8.custom_coverage.png
-rw-r--r-- 1 root root    0 Oct  8 11:23 test_3e7e8.custom.done.txt
-rw-r--r-- 1 root root 455K Oct  8 11:23 test_3e7e8.custom_pae.png
-rw-r--r-- 1 root root 237K Oct  8 11:23 test_3e7e8.custom_plddt.png
-rw-r--r-- 1 root root 1.2M Oct  8 11:23 test_3e7e8.custom_predicted_aligned_error_v1.json
-rw-r--r-- 1 root root 991K Oct  8 10:46 test_3e7e8.custom_scores_rank_001_alphafold2_multimer_v3_model_1_seed_000.json
-rw-r--r-- 1 root root 

In [23]:
!ls -lh

total 28M
-rw-r--r-- 1 root root 1020 Oct  8 11:09 cite.bibtex
-rw-r--r-- 1 root root    0 Oct  8 10:37 COLABFOLD_READY
-rw-r--r-- 1 root root 1.2K Oct  8 11:09 config.json
-rw-r--r-- 1 root root 3.2K Oct  8 11:23 log.txt
drwxr-xr-x 2 root root 4.0K Oct  8 10:54 output
drwxr-xr-x 2 root root 4.0K Oct  8 10:41 params
drwxr-xr-x 1 root root 4.0K Sep 30 13:28 sample_data
-rw-r--r-- 1 root root  441 Oct  8 11:07 test_3e7e8.csv
-rw-r--r-- 1 root root  13M Oct  8 11:09 test_3e7e8.custom.a3m
-rw-r--r-- 1 root root  85K Oct  8 11:09 test_3e7e8.custom_coverage.png
-rw-r--r-- 1 root root    0 Oct  8 11:23 test_3e7e8.custom.done.txt
-rw-r--r-- 1 root root 455K Oct  8 11:23 test_3e7e8.custom_pae.png
-rw-r--r-- 1 root root 237K Oct  8 11:23 test_3e7e8.custom_plddt.png
-rw-r--r-- 1 root root 1.2M Oct  8 11:23 test_3e7e8.custom_predicted_aligned_error_v1.json
-rw-r--r-- 1 root root 991K Oct  8 10:46 test_3e7e8.custom_scores_rank_001_alphafold2_multimer_v3_model_1_seed_000.json
-rw-r--r-- 1 root root 

In [40]:
cat test2_3e7e8.a3m

In [48]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive, try disabling your adblocker and run this cell again. If that fails click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

if msa_mode == "custom":
  print("Don't forget to cite your custom MSA generation method.")
if msa_mode != "custom":
  !cat $jobname"_env/"*".a3m" > $jobname".a3m"
!zip -FSr $jobname".result.zip" $jobname".log" $a3m_file $jobname*".pdb" $jobname*"_coverage_lDDT.png" "cite.bibtex" $jobname*"_PAE.png"
files.download(f"{jobname}.result.zip")

if save_to_google_drive == True and drive:
  uploaded = drive.CreateFile({'title': f"{jobname}.result.zip"})
  uploaded.SetContentFile(f"{jobname}.result.zip")
  uploaded.Upload()
  print(f"Uploaded {jobname}.result.zip to Google Drive with ID {uploaded.get('id')}")

cat: 'test2_3e7e8_env/*.a3m': No such file or directory
	zip warning: name not matched: test2_3e7e8*_coverage_lDDT.png
	zip warning: name not matched: test2_3e7e8*_PAE.png
updating: test2_3e7e8.log (deflated 14%)
updating: test2_3e7e8.a3m (stored 0%)
updating: test2_3e7e8_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb (deflated 79%)
updating: cite.bibtex (deflated 55%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Instructions <a name="Instructions"></a>
**Quick start**
1. Paste your protein sequence(s) in the input field.
2. Press "Runtime" -> "Run all".
3. The pipeline consists of 5 steps. The currently running steps is indicated by a circle with a stop sign next to it.

**Result zip file contents**

1. PDB formatted structures sorted by avg. pIDDT and complexes are sorted by pTMscore. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. BibTeX file with citations for all used tools and databases.

At the end of the job a download modal box will pop up with a `jobname.result.zip` file. Additionally, if the `save_to_google_drive` option was selected, the `jobname.result.zip` will be uploaded to your Google Drive.

**MSA generation for complexes**

For the complex prediction we use unpaired and paired MSAs. Unpaired MSA is generated the same way as for the protein structures prediction by searching the UniRef100 and environmental sequences three iterations each.

The paired MSA is generated by searching the UniRef100 database and pairing the best hits sharing the same NCBI taxonomical identifier (=species or sub-species). We only pair sequences if all of the query sequences are present for the respective taxonomical identifier.

**Using a custom MSA as input**

To predict the structure with a custom MSA (A3M formatted): (1) Change the msa_mode: to "custom", (2) Wait for an upload box to appear at the end of the "Input Protein ..." box. Upload your A3M. The first fasta entry of the A3M must be the query sequence without gaps.

As an alternative for MSA generation the [HHblits Toolkit server](https://toolkit.tuebingen.mpg.de/tools/hhblits) can be used. After submitting your query, click "Query Template MSA" -> "Download Full A3M". Download the A3M file and upload it in this notebook.

**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or Mgnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues


**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- [Söding Lab](https://www.mpibpc.mpg.de/soeding) for providing the computational resources for the MMseqs2 server

- Richard Evans for helping to bechmark the ColabFold's Alphafold-multimer support

- [David Koes](https://github.com/dkoes) for his awesome [py3Dmol](https://3dmol.csb.pitt.edu/) plugin, without whom these notebooks would be quite boring!

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
